# 05 — Agent-loop benchmark

**The prior notebooks measure latency, cost, and token counts. Those numbers drift
with server load, region, time of day. This notebook measures something that
ages better: can the model close a multi-step agent loop?**

Setup:

- A fixed 12-product catalog with deterministic per-dimension scores.
- 4 mock tools with production-realistic schemas (`search_products`,
  `get_reviews`, `compare`, `finalize`). Tools are pure functions of the
  catalog — no external variance.
- 18 parameterized tasks ("Recommend a {category} under ${budget} that
  maximizes {priority}"). Ground truth is a deterministic function of the
  catalog, so "correct answer" is unambiguous.
- **20% error injection** on non-finalize tool calls (seeded RNG, independent
  per provider). Measures recovery behavior — when a tool returns `{error:
  "..."}`, does the agent retry, reformulate, or give up?
- Common orchestration harness. Model call + tool-result threading are native
  to each provider (Anthropic `tools` with content-block results, OpenAI
  `tools` with `tool_call_id` results), but the loop shape is identical.

What this produces:

- **Success rate** — how often did the agent reach the ground-truth answer?
- **Mean turns to completion** — model→tool→model round-trips per successful task.
- **Cost per successful task** — unit economics after factoring in retries
  and failed runs. This is the number that actually matters for agent pricing.
- **Recovery rate** — on tasks where errors were injected, did the agent
  retry the failing tool?
- **Failure-mode taxonomy** — `wrong_answer`, `loop`, `hallucinated_tool`,
  `premature_finalize`, `never_called_tool`, `api_error`.

These numbers age well because the catalog is fixed, the tools are
deterministic, and every metric is a model-behavior property — not a
deployment-weather snapshot.

In [1]:
import os

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from openai import OpenAI

from src.agent_harness import (
    CATALOG,
    generate_tasks,
    ground_truth,
    run_anthropic_task,
    run_openai_task,
)

load_dotenv()
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
anthropic_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

OPENAI_MODEL = "gpt-5.4-mini"
ANTHROPIC_MODEL = "claude-haiku-4-5"
MAX_TURNS = 10

tasks = generate_tasks()
print(f"{len(tasks)} tasks, {len(CATALOG)} products in catalog")
print("First 3 tasks:")
for t in tasks[:3]:
    print(f"  [{t['index']}] {t['prompt']}  (truth: {ground_truth(t)})")

18 tasks, 12 products in catalog
First 3 tasks:
  [0] Recommend one headphones under $150 that maximizes noise cancellation. Use the tools to research, then call finalize() with your choice.  (truth: hp_1)
  [1] Recommend one headphones under $150 that maximizes battery life. Use the tools to research, then call finalize() with your choice.  (truth: hp_1)
  [2] Recommend one headphones under $250 that maximizes noise cancellation. Use the tools to research, then call finalize() with your choice.  (truth: hp_1)


## Run the benchmark — both providers, same tasks, same error seeds

In [2]:
outcomes: list = []

for task in tasks:
    oa = run_openai_task(openai_client, OPENAI_MODEL, task, max_turns=MAX_TURNS)
    an = run_anthropic_task(anthropic_client, ANTHROPIC_MODEL, task, max_turns=MAX_TURNS)
    outcomes.extend([oa, an])
    print(
        f"task {task['index']:>2}  "
        f"openai={oa.classification:<20} ({oa.turns}t, ${oa.cost_usd:.4f})   "
        f"anthropic={an.classification:<20} ({an.turns}t, ${an.cost_usd:.4f})"
    )

print(f"\nCollected {len(outcomes)} outcomes")

task  0  openai=success              (5t, $0.0033)   anthropic=success              (3t, $0.0061)


task  1  openai=success              (3t, $0.0020)   anthropic=success              (3t, $0.0062)


task  2  openai=success              (4t, $0.0029)   anthropic=success              (4t, $0.0089)


task  3  openai=success              (5t, $0.0039)   anthropic=success              (4t, $0.0092)


task  4  openai=success              (4t, $0.0031)   anthropic=success              (3t, $0.0073)


...

task 13  openai=success              (3t, $0.0018)   anthropic=success              (4t, $0.0077)


task 14  openai=success              (4t, $0.0029)   anthropic=success              (4t, $0.0095)


task 15  openai=success              (3t, $0.0022)   anthropic=success              (3t, $0.0071)


task 16  openai=success              (5t, $0.0039)   anthropic=success              (4t, $0.0097)


task 17  openai=success              (4t, $0.0029)   anthropic=success              (3t, $0.0075)

Collected 36 outcomes


## Aggregate summary

In [3]:
def _summary_row(records):
    n = len(records)
    successes = [r for r in records if r.classification == "success"]
    errs = sum(r.errors_encountered for r in records)
    recovs = sum(r.errors_recovered for r in records)
    cost_success = (
        sum(r.cost_usd for r in successes) / len(successes) if successes else float("nan")
    )
    cost_total = sum(r.cost_usd for r in records) / n
    return {
        "n": n,
        "success_rate": round(len(successes) / n, 3),
        "mean_turns": round(sum(r.turns for r in records) / n, 2),
        "mean_turns_on_success": round(
            sum(r.turns for r in successes) / len(successes) if successes else float("nan"), 2
        ),
        "mean_cost_per_call_usd": round(cost_total, 5),
        "mean_cost_per_success_usd": round(cost_success, 5),
        "errors_encountered": errs,
        "errors_recovered": recovs,
        "recovery_rate": round(recovs / errs, 3) if errs else float("nan"),
    }


summary = pd.DataFrame(
    [
        {"provider": "openai", "model": OPENAI_MODEL,
         **_summary_row([r for r in outcomes if r.provider == "openai"])},
        {"provider": "anthropic", "model": ANTHROPIC_MODEL,
         **_summary_row([r for r in outcomes if r.provider == "anthropic"])},
    ]
).set_index("provider")
summary

,model,n,success_rate,mean_turns,mean_turns_on_success,mean_cost_per_call_usd,mean_cost_per_success_usd,errors_encountered,errors_recovered,recovery_rate
provider,,,,,,,,,,
openai,gpt-5.4-mini,18,1.0,3.78,3.78,0.00270,0.00270,16,12,0.750
anthropic,claude-haiku-4-5,18,1.0,3.44,3.44,0.00777,0.00777,11,8,0.727


## Failure-mode breakdown

In [4]:
breakdown = pd.crosstab(
    index=[r.classification for r in outcomes],
    columns=[r.provider for r in outcomes],
    margins=True,
    margins_name="total",
).rename_axis(index="classification", columns="provider")
breakdown

provider,anthropic,openai,total
classification,,,
success,18,18,36
total,18,18,36


## Per-task detail

In [5]:
by_task = pd.DataFrame([
    {
        "task": t["index"],
        "category": t["category"],
        "budget": t["budget"],
        "priority": t["priority"],
        "truth": ground_truth(t),
        "openai_result": next(r for r in outcomes if r.task_index == t["index"] and r.provider == "openai").finalized_id,
        "openai_class": next(r for r in outcomes if r.task_index == t["index"] and r.provider == "openai").classification,
        "openai_turns": next(r for r in outcomes if r.task_index == t["index"] and r.provider == "openai").turns,
        "anthropic_result": next(r for r in outcomes if r.task_index == t["index"] and r.provider == "anthropic").finalized_id,
        "anthropic_class": next(r for r in outcomes if r.task_index == t["index"] and r.provider == "anthropic").classification,
        "anthropic_turns": next(r for r in outcomes if r.task_index == t["index"] and r.provider == "anthropic").turns,
    }
    for t in tasks
])
by_task

,task,category,budget,priority,truth,openai_result,openai_class,openai_turns,anthropic_result,anthropic_class,anthropic_turns
0,0,headphones,150,noise_cancellation,hp_1,hp_1,success,5,hp_1,success,3
1,1,headphones,150,battery_life,hp_1,hp_1,success,3,hp_1,success,3
2,2,headphones,250,noise_cancellation,hp_1,hp_1,success,4,hp_1,success,4
3,3,headphones,250,battery_life,hp_4,hp_4,success,5,hp_4,success,4
4,4,headphones,450,noise_cancellation,hp_3,hp_3,success,4,hp_3,success,3
5,5,headphones,450,battery_life,hp_3,hp_3,success,4,hp_3,success,3
6,6,laptop,800,battery_life,lp_3,lp_3,success,3,lp_3,success,3
7,7,laptop,800,performance,lp_3,lp_3,success,3,lp_3,success,3
8,8,laptop,1500,battery_life,lp_1,lp_1,success,4,lp_1,success,4
9,9,laptop,1500,performance,lp_4,lp_4,success,3,lp_4,success,4
